# 📊 4. Final Dataset Creation & Integration

Este notebook es la etapa de consolidación final del pipeline de datos de **ProtT5**. Su objetivo es fusionar la información estructural (secuencias) con la información semántica (embeddings) para generar los archivos listos para el entrenamiento del modelo.

---

## 📋 Descripción General
El script realiza una operación de **inner join** entre los archivos de metadatos CSV y los vectores numéricos generados en la fase anterior. Esto asegura que cada secuencia esté perfectamente alineada con su correspondiente representación de 1024 dimensiones.



## 🛠️ Funcionalidades Principales

1.  **Carga de Datos Multiformato:**
    * Importa los archivos CSV con identificadores y etiquetas.
    * Carga los archivos binarios de NumPy (`.npy`) que contienen los tensores de embeddings.
2.  **Alineación y Fusión (Merging):**
    * Utiliza la columna `id` (del CSV) y `seq_id` (del embedding) para unir ambos datasets.
    * Garantiza la integridad de los datos mediante una unión de tipo *inner*, eliminando cualquier inconsistencia potencial.
3.  **Estructuración Final:**
    * Filtra y organiza las columnas en el formato estándar requerido por el clasificador:
        * `seq_id`: Identificador único.
        * `sequence`: Cadena de aminoácidos.
        * `embedding`: Vector de 1024 dimensiones.
        * `label`: Clase (0 o 1).
4.  **Serialización de Objetos:**
    * Guarda los datasets finales en formato de archivos de datos de PyTorch o CSV estructurados para un acceso rápido durante el entrenamiento.

## 📐 Especificaciones del Output
* **Dimensión del Embedding:** 1024 (ProtFlash).
* **Datasets generados:** `train_dataset`, `val_dataset` y `test_dataset`.
* **Total de registros:** Sincronizado con los procesos previos (aprox. 39k-40k secuencias).

Import of libraries

In [1]:
import pandas as pd
import numpy as np
import torch

Route configuration

In [4]:
INPUT_EMBEDDINGS = "/content/drive/MyDrive/Experimentos/ProtT5/Embeddings/Generate/"
INPUT_DATA = "/content/drive/MyDrive/Experimentos/Data_Preparation/Data/csv/Dataset_Consolidation/"
OUTPUT_DATASETS = "/content/drive/MyDrive/Experimentos/ProtT5/Embeddings/Datasets/"

Sequence data loading

In [5]:
train_data = pd.read_csv(INPUT_DATA + "train.csv")
val_data = pd.read_csv(INPUT_DATA + "val.csv")
test_data = pd.read_csv(INPUT_DATA + "test.csv")

In [6]:
train_data.head()

,Unnamed: 0,id,sequence,label
0,0,train_neg_0,NMLFIKNANNVKSLIKGECTLSVEIDGKEFIINGIKIAKRPEDRVG...,0
1,1,train_neg_1,RMNESTAADLHA,0
2,2,train_neg_2,PQETNLTLKVCKPQAPQKLDPTLSSCFFIESCGAQVPVEGSRRETS...,0
3,3,train_neg_3,ALRHVTIDILDDANRQNDQLGQTVRIPGLFDGRASTYPVDIPPVGI...,0
4,4,train_neg_4,VSSRALFDLEEGNA,0


Loading of pre-generated embeddings

In [7]:
train_embeddings = pd.read_csv(INPUT_EMBEDDINGS + "train_embeddings.csv")
val_embeddings = pd.read_csv(INPUT_EMBEDDINGS + "val_embeddings.csv")
test_embeddings = pd.read_csv(INPUT_EMBEDDINGS + "test_embeddings.csv")

In [8]:
train_embeddings.head()

,seq_id,embedding
0,train_neg_0,"[0.03020147606730461, -0.06806240975856781, 0...."
1,train_neg_1,"[0.0937957763671875, -0.13228636980056763, -0...."
2,train_neg_2,"[0.03889891877770424, 0.09978769719600677, -0...."
3,train_neg_3,"[-0.04882824048399925, 0.07979049533605576, 0...."
4,train_neg_4,"[0.106353759765625, -0.10037434846162796, 0.14..."


Sequence fusion with embeddings (join by ID)

In [9]:
train_dataset = pd.merge(train_data, train_embeddings, left_on="id", right_on="seq_id",how="inner")
val_dataset = pd.merge(val_data, val_embeddings, left_on="id", right_on="seq_id",how="inner")
test_dataset = pd.merge(test_data, test_embeddings, left_on="id", right_on="seq_id",how="inner")

Selection of final columns

In [10]:
train_dataset = train_dataset[["seq_id","sequence","embedding","label"]]
val_dataset = val_dataset[["seq_id","sequence","embedding","label"]]
test_dataset = test_dataset[["seq_id","sequence","embedding","label"]]

Saving of final datasets

In [11]:
train_dataset.to_csv(OUTPUT_DATASETS+"/train_dataset.csv")
val_dataset.to_csv(OUTPUT_DATASETS+"/val_dataset.csv")
test_dataset.to_csv(OUTPUT_DATASETS+"/test_dataset.csv")